# Variation margin and capital

`SimpleMargin` is a bilateral variation-margin agreement. It has no initial margin.

For a positive mark-to-market the call uses **my** threshold: the counterparty posts down to `MtM - threshold`. For a negative mark-to-market the call uses the **counterparty** threshold. A call smaller than the minimum transfer amount is skipped. Delivery amounts are rounded away from zero; returns are rounded toward zero.

Inside the threshold the agreement returns the whole balance.

Posted collateral changes Basel II current exposure. With a positive mark-to-market, unmargined replacement cost is the mark-to-market itself. After collateral is posted, replacement cost is `max(0, MtM - balance)`.


In [1]:
import math

import sjd
from sjd import examples

print(sjd.__file__)


E:\Code\sjd\python\sjd\__init__.py


## One agreement, three mark-to-markets


In [2]:
margin = sjd.SimpleMargin(
    examples.ANCHOR,
    use_my_threshold=True,
    my_threshold=1_000.0,
    use_cpty_threshold=True,
    cpty_threshold=1_000.0,
    minimum_transfer_amount=500.0,
    rounding=100.0,
)
assert margin.is_ok()

cases = [
    ("Outside my threshold", 10_000.0, 9_000.0),
    ("Call below the minimum transfer", 1_400.0, 0.0),
    ("Outside the counterparty threshold", -10_000.0, -9_000.0),
]
print(f"{'Case':<36} {'MtM':>12} {'Call':>12}")
for label, mtm, expected in cases:
    call = margin.get_margin_call(mtm)
    print(f"{label:<36} {mtm:12,.2f} {call:12,.2f}")
    assert call == expected


Case                                          MtM         Call
Outside my threshold                    10,000.00     9,000.00
Call below the minimum transfer          1,400.00         0.00
Outside the counterparty threshold     -10,000.00    -9,000.00


## The same rules on the sample book

The CSA below is sized to the book: threshold 50,000, minimum transfer 10,000, rounding 1,000. A zero balance produces the same capital as the unmargined portfolio. Setting the balance to the mark-to-market removes the positive replacement cost and leaves the add-on.


In [3]:
market = examples.sample_market()
book = examples.sample_contracts(market["forward"], market)
environment = market["environment"]
print("anchor", market["anchor"])
print(
    "collar", market["collar_settlement"],
    "put", book["put_strike"],
    "call", book["call_strike"],
)
print("forward", market["forward_settlement"], "strike", book["forward_strike"])
print(
    "asian", market["asian_settlement"],
    "averaging days", len(market["averaging_dates"]),
    "strike", book["asian_strike"],
)

credit = examples.credit_parameters()
factors = sjd.reg_cap_factors("commodity")
bare = sjd.PortfolioB2CEM(book["contracts"], factors, credit, True)
agreement = sjd.SimpleMargin(
    market["anchor"],
    True,
    50_000.0,
    True,
    50_000.0,
    10_000.0,
    1_000.0,
)
collateralised = sjd.PortfolioB2CEM(
    examples.sample_contracts(market["forward"], market)["contracts"],
    factors,
    credit,
    True,
    agreement,
)
assert not bare.is_margined()
assert collateralised.is_margined()

for portfolio in (bare, collateralised):
    report = portfolio.validate_report(environment)
    assert report.ok, [issue.message for issue in report.issues]

mtm = collateralised.get_value(environment)
call = collateralised.get_margin_call(environment)
print(f"Mark-to-market {mtm:,.2f}")
print(f"Margin call    {call:,.2f}")
assert math.isclose(call, agreement.get_margin_call(mtm), rel_tol=1e-12)
assert mtm > 0.0

unposted = collateralised.get_regulatory_capital(environment)
assert math.isclose(unposted, bare.get_regulatory_capital(environment), rel_tol=1e-10)
collateralised.set_margin_balance(mtm)
posted = collateralised.get_regulatory_capital(environment)
print(f"Capital, balance 0     {unposted:,.2f}")
print(f"Capital, balance = MtM {posted:,.2f}")
assert posted < unposted


anchor 2026-09-16
collar 2026-12-17 put 17.1 call 18.16
forward 2027-03-16 strike 17.41
asian 2027-07-02 averaging days 20 strike 17.72
Mark-to-market 410,941.59
Margin call    361,000.00
Capital, balance 0     376,664.68
Capital, balance = MtM 343,789.35
